# Сечения фазовой траектории

## Импорты и параметры

In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.decomposition import PCA
from scipy.signal import find_peaks
from statsmodels.tsa.stattools import acf
from my_lib import HankelMatrix, prepare_time_series

pio.renderers.default = "plotly_mimetype"
np.random.seed(0)
WINDOW = 500
PERIOD = 460
N_CYCLES = 18

In [2]:
def CartesianToSpherical(point):
    r = np.sqrt(sum(point ** 2))
    n = len(point)
    phi = np.zeros(n - 1)
    
    for i in range(n - 2):
        phi[i] = np.arccos(point[i] / np.sqrt(sum(point[i:] ** 2)))
        
    if point[-1] >= 0:
        phi[n - 2] = np.arccos(point[n - 2] / np.sqrt(point[n - 1] ** 2 + point[n - 2] ** 2))
    else:
        phi[n - 2] = 2 * np.pi - np.arccos(point[n - 2] / np.sqrt(point[n - 1] ** 2 + point[n - 2] ** 2))
        
    return np.hstack((phi, r))


def TrajectoryToSpherical(tr):
    tr_spherical = np.zeros(tr.shape)
    for i, point in enumerate(tr):
        tr_spherical[i] = CartesianToSpherical(point)
    return tr_spherical


def SphericalToCartesian(point):
    phi, r = point[:-1], point[-1]
    n = len(point)
    x = np.zeros(n) 
    cur = r
    
    for i in range(n - 1):
        x[i] = cur * np.cos(phi[i])
        cur *= np.sin(phi[i])

    x[n - 1] = cur
    return x
    

def TrajectoryToCartesian(tr):
    tr_cartesian = np.zeros(tr.shape)
    for i, point in enumerate(tr):
        tr_cartesian[i] = SphericalToCartesian(point)
    return tr_cartesian

In [3]:
def orientation(point, n, point0, dist=False):
    ans = 0
    for i in range(len(n)):
        ans += n[i]*(point[i] - point0[i])
    if dist:
        return ans/(np.sum(n**2))**.5
        
    else:
        return np.sign(ans)

## Две записи ходьбы

Второй записи задаём искусственное изменение амплитуды.

In [4]:
x_acc_test, t_test = prepare_time_series('data/30 sec 2_accm.csv')
x_acc, t = prepare_time_series('data/long_walk_100_acc.csv')
x_acc = x_acc[7007:7007 + 450 * 20]

fig = go.Figure(go.Scatter(y=x_acc, mode='lines', name='Сигнал'))
fig.update_layout(template='plotly_white', height=350,
                  xaxis_title='Отсчёт', yaxis_title='Ускорение')
fig.show()

In [5]:
x_acc_test_ = x_acc_test[:25000]*0.8

In [6]:
modif = np.concatenate([np.ones(3000),
                        np.linspace(1,0,500),
                        np.zeros(1500)+.2,
                        np.linspace(0,1,1000),
                        np.ones(6000),
                        np.linspace(1,0,400),
                        np.zeros(4000)+.2,
                        np.linspace(0,1,1000)],
                        axis = 0)

In [7]:
modif = np.concatenate([modif,
                       np.ones(len(x_acc_test_)-len(modif))],
                       axis = 0)

In [8]:
x_acc_test_m = x_acc_test_ * 0.8 * modif

## Плоскость через опорную точку

Повторяющиеся события ищем по пересечениям плоскости в окрестности выбранной точки.

In [9]:
X = HankelMatrix(x_acc, WINDOW)
X_test = HankelMatrix(x_acc_test_m, WINDOW)
pca = PCA(n_components=3, svd_solver="full")
X_PCA = pca.fit_transform(X)
X_PCA_test = pca.transform(X_test)

cycles = X_PCA[:N_CYCLES * PERIOD].reshape(N_CYCLES, PERIOD, 3)
X_mean = cycles.mean(axis=0)
X_std = cycles.std(axis=0, ddof=0)
D = 3 * X_std.mean()

In [10]:
phase_index = 6060-500
Point_zero_phase = X_PCA[phase_index]
Point_half_phase = X_PCA[phase_index + 230]

In [11]:
a = X_PCA[phase_index+1]
b = X_PCA[phase_index-1]
n = (b-a)

In [12]:
indexes = []
for i in range(1,len(X_PCA_test)):
    if (np.sum((Point_zero_phase - X_PCA_test[i])**2))**.5 <= D:
        if orientation(X_PCA_test[i-1],n, Point_zero_phase) != orientation(X_PCA_test[i],n, Point_zero_phase):
            indexes.append(i)
indexes = np.array(indexes, dtype=int)

In [13]:
indexes_init = indexes + WINDOW - 1
fig = go.Figure()
fig.add_scatter(y=x_acc_test_m, mode='lines', name='Сигнал', opacity=0.5)
fig.add_scatter(x=indexes_init, y=x_acc_test_m[indexes_init],
                mode='markers', marker_size=7, name='Сечения')
fig.update_layout(template='plotly_white', height=350,
                  xaxis_title='Отсчёт', yaxis_title='Ускорение')
fig.show()

## Пересечения без ограничения радиуса

In [14]:
indexes = []
for i in range(1,len(X_PCA)):
    if (np.sum((Point_zero_phase - X_PCA[i])**2))**.5 <= np.inf:
        if orientation(X_PCA[i-1],n, Point_zero_phase) != orientation(X_PCA[i],n, Point_zero_phase):
            indexes.append(i)
indexes = np.array(indexes, dtype=int)

## Плоскость через начало координат

Модификация Мотренко. Последние два пересечения отбрасываются, как в исходном опыте.

In [15]:
x_acc, t = prepare_time_series('./data/long_walk_100_acc.csv')
x_acc = x_acc[7007:7007 + 450 * 20]
X = HankelMatrix(x_acc, WINDOW)
pca = PCA(n_components=3, svd_solver="full")
X_PCA = pca.fit_transform(X)

In [16]:
index_0 = 6060 - 500
phi_0 = X_PCA[index_0]
n_ph = phi_0.copy()

In [17]:
a = X_PCA[index_0 + 1]
b = X_PCA[index_0 - 1]
n = b - a
point_for_projection = a - 10 * n

In [18]:
t = np.sum(n_ph * point_for_projection) / np.sum(n_ph**2)
projection_point = n_ph * t

In [19]:
n_final = projection_point - point_for_projection
limit = np.sum(np.abs(X_PCA.max(axis=0) - X_PCA.min(axis=0))) / 3.5

In [20]:
cycles = X_PCA[:N_CYCLES * PERIOD].reshape(N_CYCLES, PERIOD, 3)
X_mean = cycles.mean(axis=0)
X_std = cycles.std(axis=0, ddof=0)
D = 2 * X_std.mean()

In [21]:
print(f"D:{np.round(D,3)}, Limit:{np.round(limit,3)}")

D:24.62, Limit:140.327


In [22]:
indexes = []
for i in range(1,len(X_PCA)):
    if np.sum(np.abs(phi_0 - X_PCA[i])) <= np.inf:
        if orientation(X_PCA[i-1],n_final, np.zeros(len(phi_0))) != orientation(X_PCA[i],n_final, np.zeros(len(phi_0))):
            indexes.append(i)
indexes = np.array(indexes[:-2], dtype=int)

In [23]:
indexes_init = indexes + WINDOW - 1
fig = go.Figure()
fig.add_scatter(y=x_acc, mode='lines', name='Сигнал', opacity=0.5)
fig.add_scatter(x=indexes_init, y=x_acc[indexes_init],
                mode='markers', marker_size=7, name='Сечения')
fig.update_layout(template='plotly_white', height=350,
                  xaxis_title='Отсчёт', yaxis_title='Ускорение')
fig.show()

In [24]:
np.mean(np.diff(indexes_init))

np.float64(115.16901408450704)

In [25]:
np.std(np.diff(indexes_init))

np.float64(6.331201331151638)

## Автокорреляция

Смотрим значения ACF на найденных сечениях.

In [26]:
acf_modified = acf(x_acc_test_m, nlags=len(x_acc_test_m) - 3000, fft=True)
acf_peaks, _ = find_peaks(acf_modified)
fig = go.Figure()
fig.add_scatter(y=acf_modified, mode='lines', name='ACF')
fig.add_scatter(x=acf_peaks, y=acf_modified[acf_peaks], mode='markers', name='Пики')
fig.update_layout(template='plotly_white', height=350,
                  xaxis_title='Сдвиг, отсчёты', yaxis_title='Автокорреляция')
fig.show()

In [27]:
selected_lags = np.unique(np.r_[0, indexes_init]).astype(int)
acf_clean = acf(x_acc, nlags=int(selected_lags.max()), fft=True)
acf_at_crossings = acf_clean[selected_lags]

## Сферические координаты

Углы усредняются обычным средним.

In [28]:
pca = PCA(n_components=3, svd_solver="full")
X_PCA = pca.fit_transform(HankelMatrix(x_acc, WINDOW))
X_SP = TrajectoryToSpherical(X_PCA)
cycles_sp = X_SP[:18 * 462].reshape(18, 462, 3)
X_meanSP = cycles_sp.mean(axis=0)
X_stdSP = cycles_sp.std(axis=0, ddof=0)

## Неоднозначные участки

Ищем несвязные позиции цикла, попавшие в координатный интервал среднего ± СКО.

In [29]:
pca = PCA(n_components=7, svd_solver="full")
X_PCA = pca.fit_transform(HankelMatrix(x_acc, WINDOW))
cycles = X_PCA[:N_CYCLES * PERIOD].reshape(N_CYCLES, PERIOD, 7)
X_mean = cycles.mean(axis=0)
X_std = cycles.std(axis=0, ddof=0)

In [30]:
all_indexes= []
for i in range(len(X_mean)):
    tmp = []
    x_current = X_mean[i]
    for j in range(len(X_mean)):
        if (((X_mean[j,:] + X_std[j,:]) > x_current) & ((X_mean[j,:] - X_std[j,:] < x_current))).all():
            tmp.append(j)
            
    tmp = np.array(tmp, dtype=int)
    all_indexes.append(tmp)


needed = []
for i in range(len(all_indexes)):
    if (np.diff(all_indexes[i]) > 1).any():
        needed.append(i)